# Reading from a SQL database

This module covers reading data from a relational database into Python, with SQLAlchemy Core as the connection layer and plain SQL as the query language. Readers are assumed to be comfortable with Python syntax and at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on the SELECT statements that pull source data into a tm1py script and the patterns that surround them.

Most TM1 cubes are loaded from somewhere else: an ERP, a data warehouse, a reporting database. The shape of "somewhere else" is almost always a SQL database. tm1py handles the write side; reading from the source is a separate job, and the language for that job is SQL. This module covers enough SQL to write the SELECTs that load a typical cube and enough SQLAlchemy to run them safely from Python. The ORM half of SQLAlchemy (declarative classes, sessions, relationships) is out of scope; SQLAlchemy Core is the connection and execution layer underneath the ORM, and it is all that a load script needs.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. From source database to cube: a real example
2. The Python read pattern at a glance
3. SELECT, FROM, and projections
4. Filtering rows with WHERE
5. Sorting and limiting with ORDER BY and FETCH
6. Aggregating with GROUP BY
7. Joining tables
8. Stacking results with UNION
9. Running parameterized queries from Python
10. Joining and loading the cube end to end
11. NULL, types, and the Python boundary
12. Connections, transactions, and large results
13. Real world design principles
14. Common mistakes

---

## 1. From source database to cube: a real example

A concrete scenario threads through the rest of the module. The Sales Plan cube needs an `Actuals` version each morning. The actuals come from the order management system, a SQL Server database that the operations team owns. Each night a Python script connects to that database, reads the previous day's orders, aggregates them by month, region, and product, and writes the totals into the cube.

The source database has three tables. The simplified shapes:

```
orders      (order_id, order_date, customer_id, product_id, amount)
customers   (customer_id, customer_name, country, region)
products    (product_id, product_name, category)
```

The job has four steps:

1. Connect to the source database.
2. Run a SELECT that joins orders to customers (for the region) and products (for the product name), aggregating amount by month and region and product.
3. Loop over the rows that come back.
4. Call `tm1.cells.write_values` to push the result into the cube.

The rest of the module builds up to step 2 and 3. Step 1 is a one liner once the connection string is known, and step 4 is the same `write_values` call covered in the cells module.

## 2. The Python read pattern at a glance

The simplest possible read against the source database, before any joins or aggregation, is six lines:

In [ ]:
%%python
from sqlalchemy import create_engine, text

engine = create_engine("mssql+pyodbc://etl_reader@source_dsn")

with engine.connect() as conn:
    result = conn.execute(text("SELECT order_date, amount FROM orders"))
    for row in result:
        print(row.order_date, row.amount)

What is happening, line by line. `create_engine(url)` builds an `Engine` object that knows the database driver, the host, and the credentials. The engine is a connection factory and a connection pool; it does not open a connection on its own. One engine per source database is created once at the top of the script and reused for the whole run.

`engine.connect()` checks out a real database connection from the pool. The `with` block guarantees the connection is returned to the pool when the block exits, even on an exception. SQLAlchemy returns it; it does not close it, because the next query on the same engine reuses the same physical connection.

`text("SELECT ...")` wraps a raw SQL string in a SQLAlchemy `TextClause`. This is the bridge between the SQL the database understands and the Python API around it. The string is sent to the database verbatim; SQLAlchemy adds bind parameter handling around it (Topic 9) but does not parse or rewrite the SQL.

`conn.execute(...)` sends the SQL to the database, waits for the response, and returns a `Result` object. The result is a streaming iterator over rows; the rows are not all in memory at once unless the script asks for them with `.fetchall()`.

The `for row in result` loop pulls one row at a time. Each `row` is a tuple like object whose columns can be accessed by name (`row.order_date`) or by position (`row[0]`). The names come from the SELECT list; aliases (`AS something`, see Topic 3) become the attribute names.

Connection string format varies by driver. `sqlite:///orders.db` is the simplest and works for local experimentation; `postgresql+psycopg://user:pw@host/db`, `oracle+oracledb://user:pw@host:1521/?service_name=XE`, and `mssql+pyodbc://user:pw@dsn` are the common production forms. SQLAlchemy needs the corresponding driver package installed (`psycopg`, `oracledb`, `pyodbc`); the URL prefix tells it which one.

The next six topics leave Python entirely and cover the SQL that goes inside the `text()` call.

## 3. SELECT, FROM, and projections

A SQL SELECT statement asks the database for rows from one or more tables and gets back a result set. The simplest form names the columns to return and the table to read from:

In [ ]:
SELECT order_id, order_date, amount
FROM orders;

The list between SELECT and FROM is the **projection**: it picks which columns appear in each output row. The table after FROM is the **source**. The statement returns one row per row in `orders`, with three columns each.

Two shorthands exist. `SELECT *` returns every column the table has. It is fine in interactive exploration but a poor choice in production code: it ties the script to the table's exact column layout, transfers columns the script does not need, and breaks silently when columns are added or reordered. Always list the columns explicitly in scripts that have to keep working.

Computed columns can appear in the projection alongside table columns:

In [ ]:
SELECT order_id, amount, amount * 0.19 AS vat
FROM orders;

`AS vat` gives the computed column a name (an **alias**) so it can be referenced as `row.vat` from Python. Without `AS`, the column would have a database assigned name like `?column?` or `amount * 0.19`, which is not portable and not pleasant to access from code. Aliases on plain columns work too (`SELECT order_id AS id ...`); they are the standard way to make SELECT output match the names downstream code expects.

Every statement ends with a semicolon when typed at a SQL prompt. SQLAlchemy adds it automatically if needed; it is fine to leave off in `text()` strings.

## 4. Filtering rows with WHERE

WHERE narrows the result set to rows that satisfy a condition. The condition is evaluated for each row from the source; rows where it is false (or NULL, see Topic 11) are dropped.

In [ ]:
SELECT order_id, order_date, amount
FROM orders
WHERE order_date >= '2026-04-01'
  AND order_date <  '2026-05-01';

Comparison operators are `=`, `<>` (not equal), `<`, `<=`, `>`, `>=`. Conditions combine with `AND`, `OR`, and `NOT`, with the usual precedence (AND before OR). Use parentheses when in doubt; they cost nothing and prevent the kind of bug that ships.

Pattern matching uses `LIKE` with `%` (any sequence of characters) and `_` (one character):

In [ ]:
SELECT customer_id, customer_name FROM customers
WHERE customer_name LIKE 'Acme%';

Set membership uses `IN`:

In [ ]:
SELECT customer_id, region FROM customers
WHERE region IN ('Europe', 'Americas');

Range tests use `BETWEEN`, which is inclusive on both ends:

In [ ]:
SELECT order_id FROM orders
WHERE amount BETWEEN 1000 AND 5000;

NULL is the SQL value for "unknown". Comparisons against NULL never return true; even `NULL = NULL` is NULL, not true. The right test is `IS NULL` or `IS NOT NULL`. The Python side of NULL is covered in Topic 11.

## 5. Sorting and limiting with ORDER BY and FETCH

ORDER BY sorts the result set on one or more columns. Each column can carry `ASC` (ascending, the default) or `DESC` (descending):

In [ ]:
SELECT order_id, amount
FROM orders
WHERE order_date >= '2026-04-01'
ORDER BY amount DESC, order_id ASC;

Sorting happens after WHERE filters but before any limit. The cost is non trivial on large result sets; sort only when the consumer cares about the order, and prefer letting the database sort over sorting in Python (the database has indexes, the script does not).

Limiting the row count varies by database. The standard SQL form is `FETCH FIRST n ROWS ONLY`, supported by Oracle, PostgreSQL, SQL Server, and DB2:

In [ ]:
SELECT order_id, amount
FROM orders
ORDER BY amount DESC
FETCH FIRST 10 ROWS ONLY;

PostgreSQL, MySQL, and SQLite also accept the older `LIMIT n`. SQL Server has a third form, `SELECT TOP 10 ...`, that goes between SELECT and the column list. The three forms produce the same result; pick whichever the target database supports.

LIMIT or FETCH with ORDER BY together is how "top N" queries are written. The same clause without ORDER BY returns an arbitrary subset; the database is free to pick whichever rows are cheapest, and the choice can change between runs. Always pair them.

## 6. Aggregating with GROUP BY

Aggregation collapses many rows into one summary row. The basic aggregate functions are `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`. Without GROUP BY, an aggregate over the whole result set returns one row:

In [ ]:
SELECT COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders
WHERE order_date >= '2026-04-01';

GROUP BY splits the rows into groups by the values of one or more columns, then runs the aggregates within each group. The query that drives the Sales Plan load groups by month and product:

In [ ]:
SELECT
    FORMAT(order_date, 'yyyy-MM') AS period,
    product_id,
    SUM(amount) AS total
FROM orders
WHERE order_date >= '2026-04-01' AND order_date < '2026-05-01'
GROUP BY FORMAT(order_date, 'yyyy-MM'), product_id;

`FORMAT(order_date, 'yyyy-MM')` is the SQL Server spelling for "render this date as `2026-04`". Other databases use different functions (`to_char(order_date, 'YYYY-MM')` in PostgreSQL and Oracle, `strftime('%Y-%m', order_date)` in SQLite). The function is the only line that changes; the GROUP BY structure is the same.

Two rules govern GROUP BY. First, every column in the SELECT list must either appear in the GROUP BY clause or be wrapped in an aggregate function. The database has no way to pick one value per group otherwise. Second, WHERE filters individual rows before grouping; HAVING filters whole groups after grouping:

In [ ]:
SELECT product_id, SUM(amount) AS total
FROM orders
GROUP BY product_id
HAVING SUM(amount) > 100000;

The mental order of evaluation is FROM, then WHERE, then GROUP BY, then HAVING, then SELECT (projection), then ORDER BY, then FETCH. SELECT is near the end for a reason: aliases defined there are not visible to WHERE or GROUP BY in standard SQL.

## 7. Joining tables

A JOIN combines rows from two tables based on a match condition. The most common form is `INNER JOIN`, which keeps only rows where the match succeeds on both sides:

In [ ]:
SELECT
    o.order_id,
    o.amount,
    c.country,
    c.region
FROM orders AS o
INNER JOIN customers AS c ON c.customer_id = o.customer_id;

Three things are new. First, `AS o` and `AS c` are table aliases; they shorten the column references and become required when the same column name (here, `customer_id`) exists in both tables and would otherwise be ambiguous. Second, the `ON` clause is the match condition; it can be any boolean expression but is almost always an equality between a foreign key on one side and a primary key on the other. Third, columns from both tables are now available in the SELECT list and the WHERE clause.

`LEFT JOIN` (also written `LEFT OUTER JOIN`) keeps every row from the left table, filling the right side with NULLs when there is no match:

In [ ]:
SELECT o.order_id, o.amount, c.region
FROM orders AS o
LEFT JOIN customers AS c ON c.customer_id = o.customer_id;
-- orders with a missing or unknown customer still appear, with c.region = NULL

`RIGHT JOIN` is the mirror; `FULL OUTER JOIN` keeps unmatched rows from both sides. INNER and LEFT cover the large majority of real queries.

Multiple joins chain naturally. The Sales Plan source query joins orders to customers (for the region) and products (for the name) in one statement:

In [ ]:
SELECT
    FORMAT(o.order_date, 'yyyy-MM') AS period,
    c.region,
    p.product_name,
    SUM(o.amount) AS total
FROM orders AS o
INNER JOIN customers AS c ON c.customer_id = o.customer_id
INNER JOIN products  AS p ON p.product_id  = o.product_id
WHERE o.order_date >= '2026-04-01' AND o.order_date < '2026-05-01'
GROUP BY FORMAT(o.order_date, 'yyyy-MM'), c.region, p.product_name;

This is the SELECT that the next Python topic will run.

## 8. Stacking results with UNION

UNION combines the result sets of two SELECT statements vertically: every row from the first, then every row from the second, in one result. The two sides must have the same number of columns, and the columns in each position must have compatible types; the column names of the combined result come from the first SELECT.

A common case is reading from two tables that hold the same kind of data, split for operational reasons (current year and archive, or two regions on separate servers for historical reasons):

In [ ]:
SELECT order_id, order_date, amount FROM orders_current
UNION ALL
SELECT order_id, order_date, amount FROM orders_archive;

`UNION` deduplicates the combined result, which requires sorting and is expensive. `UNION ALL` keeps every row, including duplicates, and is much faster. Use `UNION ALL` unless duplicates are actually possible and unwanted, which is rare when stacking disjoint sources.

UNION is the right tool when the two sides come from different tables. When the two sides come from the same table with different filters, a single SELECT with an OR (or with a CASE expression in the projection) is simpler and faster.

## 9. Running parameterized queries from Python

Going back to Python with the SQL knowledge in hand. The query from Topic 7 with the date range supplied from Python should be parameterized, never interpolated:

In [ ]:
%%python
from datetime import date
from sqlalchemy import create_engine, text

engine = create_engine("mssql+pyodbc://etl_reader@source_dsn")

query = text("""
    SELECT
        FORMAT(o.order_date, 'yyyy-MM') AS period,
        c.region,
        p.product_name,
        SUM(o.amount) AS total
    FROM orders AS o
    INNER JOIN customers AS c ON c.customer_id = o.customer_id
    INNER JOIN products  AS p ON p.product_id  = o.product_id
    WHERE o.order_date >= :start_date AND o.order_date < :end_date
    GROUP BY FORMAT(o.order_date, 'yyyy-MM'), c.region, p.product_name
""")

with engine.connect() as conn:
    rows = conn.execute(query, {"start_date": date(2026, 4, 1),
                                "end_date":   date(2026, 5, 1)})
    for row in rows:
        print(row.period, row.region, row.product_name, row.total)
# 2026-04 Europe   Phones    120000.0
# 2026-04 Europe   Tablets    85000.0
# 2026-04 Americas Phones    142500.0
# ...

The `:start_date` and `:end_date` placeholders are bind parameters. The dict passed as the second argument to `execute` supplies their values. Three reasons this matters:

1. **Safety.** A bind parameter cannot become SQL. A user supplied region name like `Europe'; DROP TABLE orders; --` would be a SQL injection if interpolated into the query string; as a bind parameter it is just a value the database compares to a column.
2. **Type correctness.** The driver knows that `date(2026, 4, 1)` should travel on the wire as a date, not as the string `'2026-04-01 00:00:00'`. Quoting and escaping are handled per database, not by Python string formatting.
3. **Plan reuse.** Databases cache execution plans by query text. The same query with different bind values reuses one cached plan; the same query with different interpolated dates is a different query each time, defeating the cache.

Never use f strings, `%s`, or `.format()` to build SQL with values. The only exception is identifiers (table and column names), which the database driver cannot bind; for those, validate against an allow list before substituting into the string.

## 10. Joining and loading the cube end to end

Putting the full pipeline together: read aggregated rows from the source, transform each row into a tm1py cell coordinate, write the batch to the Sales Plan cube.

In [ ]:
%%python
from datetime import date
from sqlalchemy import create_engine, text
from TM1py import TM1Service

PERIOD: dict[str, str] = {
    "2026-01": "Jan", "2026-02": "Feb", "2026-03": "Mar",
    "2026-04": "Apr", "2026-05": "May", "2026-06": "Jun",
}

source = create_engine("mssql+pyodbc://etl_reader@source_dsn")

query = text("""
    SELECT
        FORMAT(o.order_date, 'yyyy-MM') AS period,
        c.region,
        p.product_name,
        SUM(o.amount) AS total
    FROM orders AS o
    INNER JOIN customers AS c ON c.customer_id = o.customer_id
    INNER JOIN products  AS p ON p.product_id  = o.product_id
    WHERE o.order_date >= :start_date AND o.order_date < :end_date
    GROUP BY FORMAT(o.order_date, 'yyyy-MM'), c.region, p.product_name
""")

with source.connect() as conn, TM1Service(
    address="tm1.example.com", port=8001,
    user="admin", password="apple", ssl=True,
) as tm1:
    rows = conn.execute(query, {"start_date": date(2026, 4, 1),
                                "end_date":   date(2026, 5, 1)})
    cells: dict[tuple[str, ...], float] = {
        ("2026", PERIOD[row.period], row.region, row.product_name, "Actuals", "Revenue"):
            float(row.total)
        for row in rows
    }
    tm1.cells.write_values("Sales Plan", cells)

Two design notes. First, the SELECT does the heavy work: filtering, joining, and aggregating happen in the database, not in Python. The script receives one row per (period, region, product) cube cell instead of one per source order, which is typically a 100x to 10000x reduction in rows transferred. Second, the loop over `rows` builds a dict in one pass and hands the whole dict to `tm1.cells.write_values` in one call. Writing cell by cell would issue thousands of REST calls; one batched write is one REST call.

The `PERIOD` dict translates the source's `2026-04` into the cube's `Apr` element name. Source schemas and cube dimensions almost never match exactly; a small mapping table sits between them in nearly every TM1 load script.

## 11. NULL, types, and the Python boundary

SQL types and Python types are not the same set. SQLAlchemy and the underlying driver convert between them when rows arrive in Python; the conversions usually do the right thing, but a handful of cases are worth knowing.

**NULL becomes `None`.** A SQL NULL in any column comes back as Python `None`, regardless of the column's declared type. Code that does arithmetic on a column needs to handle `None` explicitly:

In [ ]:
%%python
total = sum(row.amount for row in rows if row.amount is not None)

The SQL alternative is to filter or coerce in the query: `WHERE amount IS NOT NULL` to drop rows with missing values, or `COALESCE(amount, 0)` to substitute a default.

**Numeric types.** SQL `INTEGER` becomes Python `int`. SQL `FLOAT` and `DOUBLE` become `float`. SQL `NUMERIC` and `DECIMAL` become `decimal.Decimal` by default, which is the right choice for currency but surprises code that assumes `float`. Convert explicitly at the boundary if the cube expects a float: `float(row.amount)`.

**Date and time.** SQL `DATE` becomes `datetime.date`, `TIMESTAMP` becomes `datetime.datetime`. Time zone handling depends on the driver and the column type; assume naive datetimes unless the column is declared `TIMESTAMP WITH TIME ZONE`. For TM1 element names like `"Jan"` or `"2026"`, the script formats the date into the right string form (Topic 10's `PERIOD` table).

**Strings.** SQL `VARCHAR`, `NVARCHAR`, and `TEXT` become Python `str`. The driver decodes the bytes from the database's character set; UTF 8 on the connection is the safe default to set explicitly.

**Booleans.** SQL `BOOLEAN` becomes Python `bool`. SQL Server, which has no native boolean, uses `BIT` (0 or 1); the driver typically returns `int` for those columns. `bool(row.flag)` is the explicit conversion.

The general rule: trust the driver for round trips through one database, and coerce explicitly at the boundary into TM1, where every element name is a string and every cell value is a number.

## 12. Connections, transactions, and large results

A SQLAlchemy `Engine` is a connection pool, not a single connection. Creating it is cheap; opening a real database connection is not. The pattern is one engine per source database, created at module import or script start, and reused for the lifetime of the script. Connections are checked out from the pool with `engine.connect()` and returned to it (not closed) when the `with` block exits.

In [ ]:
%%python
engine = create_engine("mssql+pyodbc://etl_reader@source_dsn", pool_size=5)

with engine.connect() as conn:
    rows = conn.execute(text("SELECT ..."))
    # use rows here
# connection returned to the pool here

For read only work, the default autocommit behavior is fine: each `execute` runs in its own transaction and commits when it returns. For inserts and updates (out of scope for this module), wrap the writes in `with engine.begin() as conn:` to get an explicit transaction that commits at the end of the block and rolls back on any exception.

For result sets that do not fit in memory (millions of rows), iterate the result directly rather than materializing it with `.fetchall()`. SQLAlchemy streams rows from the driver as they are needed:

In [ ]:
%%python
with engine.connect() as conn:
    result = conn.execution_options(stream_results=True).execute(text("SELECT ..."))
    for row in result:
        # only a chunk is in memory at any moment
        ...

`.fetchall()` reads the whole result into a list; `.fetchmany(n)` reads `n` rows at a time; iterating the result reads one at a time. For a TM1 load that aggregates in the database, the result is usually small (one row per cube cell) and `.fetchall()` is fine; for an unaggregated dump, streaming matters.

## 13. Real world design principles

**Push work into the database.** Filter with WHERE, aggregate with GROUP BY, join with JOIN. The database has indexes, statistics, and a query planner; Python has a `for` loop. A SELECT that returns one row per cube cell is faster to ship over the network and faster to load into TM1 than a SELECT that returns one row per source order and aggregates in Python.

**Always parameterize values.** No f strings, no `%s`, no `.format()` for values. Bind parameters everywhere, every time. The cost of doing it consistently is zero; the cost of getting it wrong once is a SQL injection, a quoting bug, or a plan cache miss.

**Name your columns explicitly.** Never `SELECT *` in a script that has to keep working. The list of columns is the contract between the script and the table; making it explicit makes that contract visible in code review and stable across schema changes.

**One Engine per source, reused.** Engines are connection pools; creating a new engine per query throws away the pool and pays the connection cost every time. Create the engine once at module level or pass it as a dependency.

**Coerce types at the boundary.** SQLAlchemy returns the type the driver gave it (often `Decimal` for `NUMERIC`, `date` for `DATE`). Convert to the type the cube expects (`float`, `str`) at the boundary, not deep inside the loop where a single missed conversion produces a confusing tm1py error.

**Let the database sort.** ORDER BY in the SELECT is almost always faster than `sorted()` in Python, because the database has indexes and can stop early when paired with FETCH FIRST. Sort in Python only when the data is already in memory for another reason.

**Keep the SQL readable.** Format multi line SQL inside `text("""...""")` with one clause per line and consistent indentation. The query is the most important part of the script; treat it like code, not a string.

## 14. Common mistakes

**Interpolating values into SQL.**

In [ ]:
%%python
# Wrong
region = user_input
conn.execute(text(f"SELECT * FROM customers WHERE region = '{region}'"))

# Correct
conn.execute(
    text("SELECT customer_id FROM customers WHERE region = :region"),
    {"region": region},
)

**Using `=` to test for NULL.**

In [ ]:
-- Wrong
SELECT * FROM orders WHERE customer_id = NULL;   -- always returns zero rows

-- Correct
SELECT * FROM orders WHERE customer_id IS NULL;

**Mixing aggregates and bare columns without GROUP BY.**

In [ ]:
-- Wrong
SELECT region, SUM(amount) FROM orders;          -- error or undefined behavior

-- Correct
SELECT region, SUM(amount) FROM orders GROUP BY region;

**Using FETCH FIRST or LIMIT without ORDER BY for top N.**

In [ ]:
-- Wrong
SELECT order_id, amount FROM orders FETCH FIRST 10 ROWS ONLY;
-- arbitrary 10 rows; not the largest, not the most recent, not anything

-- Correct
SELECT order_id, amount FROM orders ORDER BY amount DESC FETCH FIRST 10 ROWS ONLY;

**Calling `fetchall()` on a multi million row result.**

In [ ]:
%%python
# Wrong
rows = conn.execute(text("SELECT * FROM orders")).fetchall()   # whole table in memory

# Correct
result = conn.execution_options(stream_results=True).execute(
    text("SELECT order_id, order_date, amount FROM orders")
)
for row in result:
    process(row)

**Creating a new engine per query.**

In [ ]:
%%python
# Wrong
def read_orders():
    engine = create_engine("...")            # new pool every call
    with engine.connect() as conn:
        return conn.execute(text("SELECT ...")).fetchall()

# Correct
engine = create_engine("...")                # module level, created once

def read_orders():
    with engine.connect() as conn:
        return conn.execute(text("SELECT ...")).fetchall()

**Writing one row at a time into TM1 instead of batching.**

In [ ]:
%%python
# Wrong
for row in rows:
    tm1.cells.write_value(
        float(row.total), "Sales Plan",
        ("2026", PERIOD[row.period], row.region, row.product_name, "Actuals", "Revenue"),
    )
    # one REST call per cell; thousands of round trips

# Correct
cells = {
    ("2026", PERIOD[row.period], row.region, row.product_name, "Actuals", "Revenue"):
        float(row.total)
    for row in rows
}
tm1.cells.write_values("Sales Plan", cells)    # one REST call